In [ ]:
#Name: K.Jnanendra
#Roll no: ch.sc.u4cse24217

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

In [ ]:
CSV_FILE = "disease_dataset.csv"
TARGET_COLUMN = "susceptible"  # Change to the actual target column in your CSV.
df = pd.read_csv(CSV_FILE)
df.columns = [c.strip() for c in df.columns]
print("Columns:", df.columns.tolist())
if TARGET_COLUMN not in df.columns:
    raise ValueError(f"Target column '{TARGET_COLUMN}' not found. Set TARGET_COLUMN to the binary outcome column.")
df = df.dropna(subset=[TARGET_COLUMN]).copy()
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]
if y.dtype == "object":
    normalized = y.astype(str).str.strip().str.lower()
    mapping = {"yes": 1, "y": 1, "true": 1, "positive": 1, "susceptible": 1, "1": 1,
               "no": 0, "n": 0, "false": 0, "negative": 0, "not susceptible": 0, "0": 0}
    mapped = normalized.map(mapping)
    if mapped.isna().any():
        raise ValueError(f"Unrecognized target labels: {sorted(normalized.unique())}. Map labels to 0 and 1.")
    y = mapped.astype(int)
else:
    if y.nunique() != 2:
        raise ValueError(f"Expected binary target, found labels: {sorted(y.unique())}")
    y = y.astype(int)

numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]
transform = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]), categorical_cols)
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = Pipeline([("preprocessing", transform), ("classifier", LogisticRegression(max_iter=1000))])
model.fit(X_train, y_train)
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]
print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print("Classification report:\n", classification_report(y_test, pred, zero_division=0))
print("ROC-AUC:", round(roc_auc_score(y_test, prob), 4))

In [ ]:
print("Example input:\n", X.iloc[[0]])
print("Predicted class (0=no, 1=yes):", int(model.predict(X.iloc[[0]])[0]))
print("Estimated probability of class 1:", round(float(model.predict_proba(X.iloc[[0]])[0, 1]), 4))
print("Educational use only; not a medical diagnosis.")